In [0]:
import pandas as pd
import numpy as np
import time

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, rand, row_number, ceil, count, date_format, mean, lit
from pyspark.sql.window import Window

movielens_base = "/Volumes/workspace/default/movielens/"


In [0]:
# get the rating 
rating_schema = StructType([
    StructField("user_id", IntegerType(), nullable=False),
    StructField("movie_id", IntegerType(), nullable=False),
    StructField("rating", DoubleType(), nullable=True),
    StructField("rating_timestamp", TimestampType(), nullable=True)
])

rating = spark.read.csv(movielens_base + "rating.csv", header=True, schema=rating_schema, mode="FAILFAST", timestampFormat="yyyy-MM-dd HH:mm:ss")

display(rating.withColumn("rating_timestamp", date_format("rating_timestamp", "yyyy-MM-dd HH:mm:ss")).limit(5))



In [0]:
# Serverless compute in Databricks doesn't support cache()/localCheckpoint().
# The data is small, so collecting and recreating the DataFrame breaks the lineage.
def materialize(df):
    return spark.createDataFrame(df.toPandas(), schema=df.schema)

# rating_100_users
#      ↓
# user_id_ / movie_id_ remapping
#      ↓
# rating_10_users_np
#      ↓
# user_movie = [user_id_, movie_id_]
#      ↓
# initialize Q

rng = np.random.default_rng(seed=42)          # seed makes it reproducible

k=5
I = np.eye(k) # unity matrix kxk
lambda_ = 4
delta_E_min = 0.001
max_loop_count = 2000
max_user_cnt = 100
training_fraction = 0.8

# Select the first 100 users from MovieLens data + add a random column for training and validation split
rating_100_users = (
    rating
    .filter(col("user_id").between(1, max_user_cnt))
    .orderBy("user_id", "movie_id")
)

# User mapping: original user_id -> consecutive user_id_
user_mapping = (
    rating_100_users
    .select("user_id")
    .distinct()
    .withColumn(
        "user_id_",
        row_number().over(Window.orderBy("user_id")) - 1
    )
)

# Movie mapping: original movie_id -> consecutive movie_id_
movie_mapping = (
    rating_100_users
    .select("movie_id")
    .distinct()
    .withColumn(
        "movie_id_",
        row_number().over(Window.orderBy("movie_id")) - 1
    )
)

# Add the consecutive IDs to the ratings
rating_100_users = (
    rating_100_users
    .join(user_mapping, on="user_id", how="left")
    .join(movie_mapping, on="movie_id", how="left")
    .orderBy("user_id_", "movie_id_")
)


# Randomly rank ratings separately for each user
# assign a random order to the ratings by user to select the first training_fraction of ratings for training
# and the remaining 1-training_fraction of ratings for validation later on
w = Window.partitionBy("user_id").orderBy(rand(seed=42))

rating_100_users = rating_100_users.withColumn(
    "rating_rank", row_number().over(w)
)

# Count the total number of ratings for each user
w_user = Window.partitionBy("user_id")

rating_100_users = rating_100_users.withColumn(
    "user_rating_count",
    count("*").over(w_user)
)

# ceil(user_rating_count * training_fraction) means the training set gets the first ceil(training_fraction * total of ratings) ranks. 
# The remaining ratings go to validation.
rating_100_users = rating_100_users.withColumn(
    "validation",
    col("rating_rank") > ceil(col("user_rating_count") * training_fraction)
)

# Evaluate the random split once, so training and validation are each other's complement
rating_100_users = materialize(rating_100_users)

# Create the training and validation datasets
rating_100_users_train = rating_100_users.filter(~col("validation"))
rating_100_users_val = rating_100_users.filter(col("validation"))

rating_100_users_np_train = (
    rating_100_users_train
    .select("user_id_", "movie_id_", "rating")
    .toPandas()
    .to_numpy()
)

rating_100_users_np_val = (
    rating_100_users_val
    .select("user_id_", "movie_id_", "rating")
    .toPandas()
    .to_numpy()
)

# measure execution time
start = time.perf_counter()

user_cnt = user_mapping.count()
movie_cnt = movie_mapping.count()

print(f"user_cnt = {user_cnt}")
print(f"movie_cnt = {movie_cnt}")

# Group training ratings by user
ratings_by_user = []
for u in range(user_cnt):
    mask = rating_100_users_np_train[:, 0] == u
    movie_ids = rating_100_users_np_train[mask, 1].astype(int)
    ratings = rating_100_users_np_train[mask, 2]
    ratings_by_user.append((movie_ids, ratings))

# Group training ratings by movie
ratings_by_movie = []
for i in range(movie_cnt):
    mask = rating_100_users_np_train[:, 1] == i
    user_ids = rating_100_users_np_train[mask, 0].astype(int)
    ratings = rating_100_users_np_train[mask, 2]
    ratings_by_movie.append((user_ids, ratings))

# fill movie array with random numbers - small values
Q = rng.uniform(-0.1, 0.1, (movie_cnt, k))
P = np.zeros((user_cnt, k)) # create P

user_movie = rating_100_users_np_train[:, :2]

# initialze variable for loop
E = 1000
E_cur = 900
loop_count = 0
error_history = []

while (abs(E - E_cur) > delta_E_min) and (loop_count < max_loop_count):
    E = E_cur
    
    # loop users → calculate P
    #      ↓
    # loop movies → calculate Q
    #      ↓
    # Calulate estimation error

    # loop over all users
    for u in range(user_cnt):
        movie_ids_u, r_u = ratings_by_user[u]

        q_u = Q[movie_ids_u, :] # get the movie factors for the current user
        
        p_u = np.linalg.solve(
            q_u.T @ q_u + lambda_ * I,
            q_u.T @ r_u
        )

        #alternative: p_u= np.dot(np.linalg.inv(np.dot(q_u.T,q_u) + lambda_ * I),np.dot(q_u.T, r_u))
                                                        
        # store user profile
        P[u, :] = p_u
        
    # loop over all movies
    for i in range(movie_cnt):
        user_ids_i, r_i = ratings_by_movie[i]

        p_i = P[user_ids_i, :] # user-factor matrix for the users who rated movie i

        q_i = np.linalg.solve(
                p_i.T @ p_i + lambda_ * I,
                p_i.T @ r_i
            )
        
        # store user profile
        Q[i,:] = q_i

    user_movie = rating_100_users_np_train[:, :2] #selects all rows, but only the first two columns of the NumPy array.

    # calulate the estimation error
    E_cur = 0
    for u in range(user_cnt):
        movie_mask = user_movie[:, 0] == u # getthe mask for the current user just to keep the rated movies 
        movie_ids_u = user_movie[movie_mask, 1].astype(int)

        q_u = Q[movie_ids_u, :] # get the movie factors for the current user
        r_u = rating_100_users_np_train[movie_mask,2]

        e_u = np.linalg.norm(r_u - q_u @ P[u, :].T) # calculate the error for the current user
        # print(f"Error for user {u} = {e_u}") 
        E_cur += e_u**2 # the function np.linalg.norm gives the square root, so we need to square it 

    E_cur += lambda_ * (np.linalg.norm(P,'fro')**2 + np.linalg.norm(Q,'fro')**2) 
    loop_count += 1
    print(f"Error = {E_cur}")
    error_history.append((loop_count, float(E_cur)))

print(f"Total error = {E_cur}.  Loop count = {loop_count}")

print(
    f"Mean rating training data = "
    f"{rating_100_users_train.select(mean('rating')).collect()[0][0]:.4f}")
# Extract user IDs, movie IDs and actual ratings from training data
user_ids = rating_100_users_np_train[:, 0].astype(int)
movie_ids = rating_100_users_np_train[:, 1].astype(int)
actual_ratings = rating_100_users_np_train[:, 2]

# Predict ratings using the dot product of user and movie factors
predicted_ratings = np.sum(P[user_ids] * Q[movie_ids], axis=1)

# Calculate training data RMSE
rmse = np.sqrt(np.mean((actual_ratings - predicted_ratings)**2))

print(f"Training data RMSE = {rmse:.4f}")

elapsed = time.perf_counter() - start
print(f"ALS training time: {elapsed:.2f} seconds")


In [0]:
## Test prediction on validation dataset

print(
    f"Mean rating validation data = "
    f"{rating_100_users_val.select(mean('rating')).collect()[0][0]:.4f}"
)

# Count training ratings per user and per movie
user_train_counts = np.bincount(
    rating_100_users_np_train[:, 0].astype(int),
    minlength=user_cnt
)
movie_train_counts = np.bincount(
    rating_100_users_np_train[:, 1].astype(int),
    minlength=movie_cnt
)

# Extract user IDs, movie IDs and actual ratings from validation data
user_ids = rating_100_users_np_val[:, 0].astype(int)
movie_ids = rating_100_users_np_val[:, 1].astype(int)
actual_ratings = rating_100_users_np_val[:, 2]

# Keep only validation ratings for movies that have at least one training rating.
# Movies that only appear in validation keep a zero factor vector and would be
# predicted as 0. Spark excludes them (null prediction from the left join),
# so they are excluded here as well to make both RMSEs comparable.
known = movie_train_counts[movie_ids] > 0
print(f"Validation ratings without prediction = {(~known).sum()} of {len(known)}")

user_ids = user_ids[known]
movie_ids = movie_ids[known]
actual_ratings = actual_ratings[known]

# Predict ratings using the dot product of user and movie factors
predicted_ratings = np.sum(P[user_ids] * Q[movie_ids], axis=1)

# Calculate validation RMSE
rmse = np.sqrt(np.mean((actual_ratings - predicted_ratings)**2))

print(f"Validation RMSE = {rmse:.4f}")
print(f"Mean predicted rating = {predicted_ratings.mean():.4f}")
print(f"Min predicted rating  = {predicted_ratings.min():.4f}")
print(f"Max predicted rating  = {predicted_ratings.max():.4f}")

# Calculate absolute prediction errors
errors = np.abs(actual_ratings - predicted_ratings)

# Show the 10 largest errors on the validation data
worst = np.argsort(errors)[-10:][::-1]

for j in worst:
    u = user_ids[j]
    i = movie_ids[j]

    print(
        f"user={u} (training ratings={user_train_counts[u]}), "
        f"movie={i} (training ratings={movie_train_counts[i]}), "
        f"actual={actual_ratings[j]:.1f}, "
        f"predicted={predicted_ratings[j]:.2f}, "
        f"error={errors[j]:.2f}"
    )

    print(
        f"    ||P[u]||={np.linalg.norm(P[u]):.3f}, "
        f"||Q[i]||={np.linalg.norm(Q[i]):.3f}"
    )

In [0]:
# save tables to make them available for visualisation
from pyspark.sql.functions import lit, col

tbl = "workspace.default"

(spark.createDataFrame(error_history, ["iteration", "objective"])
    .select(
        col("iteration").cast("long"),
        col("objective").cast("double"),
    )
    .withColumn("implementation", lit("numpy"))
    .write.mode("overwrite")
    .option("replaceWhere", "implementation = 'numpy'")
    .saveAsTable(f"{tbl}.als_convergence"))


# Build the validation predictions directly from the validation data,
# so the result does not depend on which cell ran last.
val_user_ids = rating_100_users_np_val[:, 0].astype(int)
val_movie_ids = rating_100_users_np_val[:, 1].astype(int)
val_actual = rating_100_users_np_val[:, 2].astype(float)

# Exclude movies without training ratings (same as in the validation RMSE)
movie_train_counts = np.bincount(
    rating_100_users_np_train[:, 1].astype(int),
    minlength=movie_cnt
)
known = movie_train_counts[val_movie_ids] > 0

val_pdf = pd.DataFrame({
    "user_id": val_user_ids[known],
    "movie_id": val_movie_ids[known],
    "rating": val_actual[known],
    "prediction": np.sum(P[val_user_ids[known]] * Q[val_movie_ids[known]], axis=1),
})


(spark.createDataFrame(val_pdf)
    .select(
        col("user_id").cast("long"),
        col("movie_id").cast("long"),
        col("rating").cast("double"),
        col("prediction").cast("double"),
    )
    .withColumn("implementation", lit("numpy"))
    .write.mode("overwrite")
    .option("replaceWhere", "implementation = 'numpy'")
    .saveAsTable(f"{tbl}.als_validation"))

In [0]:
print(f"Total error = {E_cur}.  Loop count = {loop_count}")

print(
    f"Mean rating training data = "
    f"{rating_100_users_train.select(mean('rating')).collect()[0][0]:.4f}")

# Extract user IDs, movie IDs and actual ratings from training data
user_ids = rating_100_users_np_train[:, 0].astype(int)
movie_ids = rating_100_users_np_train[:, 1].astype(int)
actual_ratings = rating_100_users_np_train[:, 2]

# Predict ratings using the dot product of user and movie factors
predicted_ratings = np.sum(P[user_ids] * Q[movie_ids], axis=1)

# Calculate validation RMSE
rmse = np.sqrt(np.mean((actual_ratings - predicted_ratings)**2))

print(f"Training data RMSE = {rmse:.4f}")

print(f"||P|| max = {np.linalg.norm(P, axis=1).max():.4f}")
print(f"||Q|| max = {np.linalg.norm(Q, axis=1).max():.4f}")